In [ ]:
!wget -q -c -O rev1-benchmark-data.tar.gz https://github.com/huboqiang/omnialigner/releases/download/rev1/rev1-benchmark-data.tar.gz
!git clone --quiet --depth 1 --branch rev1 https://github.com/huboqiang/omnialigner.git omnialigner-rev1

from pathlib import Path
import hashlib, tarfile, sys
assert (3, 10) <= sys.version_info[:2] < (3, 12), "Use a Python 3.10 or 3.11 Jupyter kernel."
archive = Path("rev1-benchmark-data.tar.gz").resolve()
checkout = Path("omnialigner-rev1").resolve()
assert archive.is_file() and (checkout / ".git").is_dir(), "Download or clone failed; inspect the messages above."
assert hashlib.sha256(archive.read_bytes()).hexdigest() == "9b98464f0b6625bae48514c601b740caee50aeb008525f103a362fc7a630d7d3", "Data archive checksum mismatch"
destination = checkout / "benchmarks/rev1"
with tarfile.open(archive, "r:gz") as bundle:
    for member in bundle.getmembers():
        if member.name == "NCS" or member.name.startswith("NCS/"):
            member.name = "OmniAligner" + member.name[3:]
        target = (destination / member.name).resolve()
        assert target.is_relative_to(destination) and not (member.issym() or member.islnk()), "Unsafe archive member"
    bundle.extractall(destination)
PACKAGE_ROOT = destination / "OmniAligner"
print("Downloaded and verified rev1 data; package:", PACKAGE_ROOT.name)


In [ ]:
%pip install -q -r {PACKAGE_ROOT / "requirements.txt"}
%cd {PACKAGE_ROOT}


## Download and run
Use a **fresh Python 3.10/3.11 Jupyter kernel** with `wget` and `git` available. The first two commands download the public data archive and clone the `rev1` branch. No private directory or credentials are required. The six-dataset scoring run is CPU-based; allow 48–64 GB RAM for the largest dataset. The table-only and DLPFC examples are smaller.


# Reproduce the complete NCS submission table and ranks

**Main entry for the manuscript table**, matching the current `07_NCS_updated` comparison. This reconstructs every included method row and all 36 rank/count cells; it is not limited to OmniAligner.

Numerical sources are explicit: OmniAligner uses fresh release-expression + selected-coordinate recalculation; completed added methods use complete per-pair scoring files; historical baselines use the archived scored baseline records. This notebook reconstructs the full table and ranking, **not a new fitting or raw-coordinate rerun of every comparator**. Missing, failed and diagnostic entries remain visible and are not promoted into ranked results. MPC/E14–16 use the archived common-cell cohort for added scores; MLN PASTE-family results use a 5,000-cell-per-section support rather than the full-cell support of OmniAligner. These support differences remain disclosed.

The reference submission table is used only to verify the reconstructed output. Its values never fill missing calculations. The snapshot and pending conditions are printed below. Once a pending run is scored, freeze the revised submission references and rebuild this notebook's manifest before submission.

## 1. Locate the frozen table sources
Install the pinned environment described in the repository README. To rerun the OmniAligner coordinates rather than reuse this newly validated pair-score run, execute `OmniAligner_NCS_reproduce.ipynb` first. No aligned H5AD export or registration refitting is needed.

In [1]:
from pathlib import Path
import os, json, csv, math, hashlib
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from IPython.display import display, Markdown, HTML
from html import escape

def locate_repository():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks/OmniAligner_NCS_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("Run from this repository or its notebooks directory")
REPO_ROOT = locate_repository()
DATA_ROOT = REPO_ROOT / "data"
SAB_ROOT = REPO_ROOT
SOURCE_ROOT = DATA_ROOT / "sources"
OUTPUT_ROOT = REPO_ROOT / "results" / "table"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
METRICS = ["PCC", "CosSim", "SSIM", "MI", "MSE", "Region"]
NATIVE_KEYS = ["PCC", "CosSim", "SSIM", "MI", "Expression_MSE_native_ASSD_bygrid", "Accuracy"]
ATOL = 1e-10
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
manifest = json.loads((REPO_ROOT / "notebooks/NCS_submission_manifest.json").read_text())
for filename, key in [("submission_scope.json", "scope_file_sha256"), ("archived_baselines.csv", "archived_baselines_sha256"),
                      ("submission_reference.csv", "reference_table_sha256"), ("submission_ranks_reference.csv", "reference_ranks_sha256")]:
    assert sha256(SOURCE_ROOT / filename) == manifest[key], filename
print("Submission candidate snapshot:", manifest["snapshot"])
print("Pending conditions:", manifest["pending"])
display(pd.DataFrame(manifest["bindings"])[["dataset", "method", "kind", "status"]])

Submission candidate snapshot: 2026-11-17T01:07:58-07:00
Pending conditions: []


,dataset,method,kind,status
0,DLPFC,OmniAligner,fresh_coordinate_recalculation,Scored
1,DLPFC,CAST,archived_scored_workbook,Scored
2,DLPFC,PASTE,archived_scored_workbook,Scored
3,DLPFC,PASTE2,archived_scored_workbook,Scored
4,DLPFC,PASTE (p0),archived_scored_workbook,Scored
5,DLPFC,SANTO,archived_scored_workbook,Scored
6,DLPFC,SLAT,archived_scored_workbook,Scored
7,DLPFC,SPACEL,archived_scored_workbook,Scored
8,DLPFC,STalign,archived_scored_workbook,Scored
9,DLPFC,MOSAICField (RNA),archived_scored_workbook,Scored


## 2. Reconstruct every numerical entry
Gene/pair definitions are inherited from the frozen original scoring records. Pair means require every expected pair and finite values for that metric. Insufficient-grid default molecular scores are displayed as NA; Region is retained separately. Historical workbook values are labelled as archived scored results rather than newly recomputed coordinates.

In [2]:
baseline_rows = {(r["Dataset"], r["Method"]): r for r in csv.DictReader((SOURCE_ROOT / "archived_baselines.csv").open())}
csv.field_size_limit(256 * 1024 * 1024)
def finite_or_none(value):
    try:
        value = float(value)
        return value if math.isfinite(value) else None
    except (ValueError, TypeError):
        return None
def complete_pair_means(binding):
    path = SAB_ROOT / binding["pair_metrics_path"]
    assert sha256(path) == binding["pair_metrics_sha256"], path
    if "summary_path" in binding:
        assert sha256(SAB_ROOT / binding["summary_path"]) == binding["summary_sha256"]
    rows = list(csv.DictReader(path.open()))
    if binding["kind"] == "fresh_coordinate_recalculation":
        fresh = REPO_ROOT / "results/omni" / binding["dataset"] / "pair_metrics.csv"
        if fresh.exists():
            fresh_rows = list(csv.DictReader(fresh.open()))
            assert len(fresh_rows) == len(rows)
            for saved, recalculated in zip(rows, fresh_rows):
                assert int(saved["pair_index"]) == int(recalculated["pair_index"])
                for key in NATIVE_KEYS:
                    assert np.isclose(float(saved[key]), float(recalculated[key]), atol=ATOL, rtol=0, equal_nan=True)
            rows = fresh_rows
    assert sorted(int(row["pair_index"]) for row in rows) == list(range(binding["pairs"]))
    fallback = any(row.get("native_insufficient_grid_fallback", "False") == "True" for row in rows)
    result = []
    for index, key in enumerate(NATIVE_KEYS):
        values = [finite_or_none(row[key]) for row in rows]
        value = math.fsum(values) / len(values) if all(v is not None for v in values) else None
        result.append(None if fallback and index < 5 else value)
    return result

rebuilt_rows = []
for binding in manifest["bindings"]:
    if binding["kind"] in ["fresh_coordinate_recalculation", "completed_pair_scores"]:
        values = complete_pair_means(binding)
    elif binding["kind"] == "archived_scored_workbook":
        values = [finite_or_none(baseline_rows[(binding["dataset"], binding["method"])][metric]) for metric in METRICS]
    else:
        assert binding["kind"] == "not_scored"
        values = [None] * 6
    rebuilt_rows.append({"Dataset": binding["dataset"], "Method": binding["method"], **dict(zip(METRICS, values)),
                         "Status": binding["status"], "Eligible": binding["eligible"]})
rebuilt = pd.DataFrame(rebuilt_rows)
assert not rebuilt["Method"].isin(["Loki", "CPD"]).any()
assert rebuilt[rebuilt.Method.eq("OmniAligner")].groupby("Dataset").size().eq(1).all()

def display_table(frame, cell_style, numeric_columns=()):
    parts = ['<table style="border-collapse:collapse;font:14px sans-serif"><thead><tr><th></th>']
    parts += [f'<th style="padding:8px">{escape(str(col))}</th>' for col in frame.columns]
    parts.append('</tr></thead><tbody>')
    for label, row in frame.iterrows():
        parts.append(f'<tr><th style="padding:8px;text-align:left">{escape(str(label))}</th>')
        for col in frame.columns:
            value = row[col]
            text = 'NA' if pd.isna(value) else f'{float(value):.6f}' if col in numeric_columns else str(value)
            parts.append(f'<td style="padding:8px;border:2px solid white;{cell_style(row, col)}">{escape(text)}</td>')
        parts.append('</tr>')
    parts.append('</tbody></table>')
    display(HTML(''.join(parts)))

for dataset, group in rebuilt.groupby("Dataset", sort=False):
    display(Markdown(f"### {dataset}"))
    table = group.set_index("Method")
    best = {metric: (table.loc[table.Eligible, metric].min() if metric == "MSE" else table.loc[table.Eligible, metric].max()) for metric in METRICS}
    def cell_style(row, col):
        return "background:#d2e8dc;color:#225b43;font-weight:bold" if row.Eligible and col in METRICS and pd.notna(row[col]) and row[col] == best[col] else "background:#f1f3f4;color:#243447"
    display_table(table[METRICS + ["Status", "Eligible"]], cell_style, METRICS)

### DLPFC

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.547930,0.783503,0.531970,0.299724,0.127579,0.884759,Scored,True
CAST,0.546627,0.781952,0.527965,0.313248,0.128917,0.869842,Scored,True
PASTE,0.544071,0.778515,0.528263,0.306357,0.125379,0.838851,Scored,True
PASTE2,0.543163,0.781307,0.528125,0.304160,0.127796,0.862013,Scored,True
PASTE (p0),0.540429,0.779052,0.526039,0.285680,0.128686,0.843411,Scored,True
SANTO,0.406315,0.723795,0.396180,0.177750,0.203190,0.496139,Scored,True
SLAT,0.536767,0.778471,0.521766,0.298667,0.133502,0.834437,Scored,True
SPACEL,0.542308,0.780266,0.525264,0.298687,0.120631,0.872916,Scored,True
STalign,0.531650,0.780250,0.515869,0.269830,0.144726,0.768149,Scored,True
MOSAICField (RNA),0.293491,0.681699,0.286240,0.128547,0.273680,0.425441,Scored,True


### MHPR

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.540881,0.869558,0.527731,0.252730,1.218373,0.759039,Scored,True
PASTE,0.014652,0.733618,0.026673,0.046316,4.015312,0.177458,Scored,True
PASTE (p0),0.504022,0.855469,0.492768,0.222687,1.538849,0.721494,Scored,True
SANTO,0.080521,0.197450,0.079817,0.026639,75.695452,0.136740,Scored,True
SLAT,0.510049,0.857475,0.497675,0.231305,1.490035,0.731335,Scored,True
SPACEL,0.528168,0.863517,0.515406,0.241713,1.307417,0.752253,Scored,True
STalign,0.000000,0.000000,0.000000,0.000000,100.000000,0.000000,Diagnostic; not ranked,False
Spateo nonrigid,0.535648,0.866232,0.521511,0.249683,1.281545,0.753327,Scored,True
Spateo rigid,0.536942,0.866585,0.522419,0.251475,1.267570,0.753625,Scored,True
PASTE2,0.521702,0.861287,0.509315,0.243140,1.393879,0.741250,Scored,True


### MMPC

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.197483,0.666920,0.193019,0.069390,81.048677,0.871546,Scored,True
PASTE,0.172212,0.650976,0.170476,0.067978,81.590440,0.828582,Scored,True
SANTO,0.099983,0.606989,0.104715,0.045556,93.775101,0.721195,Scored,True
SLAT,0.183108,0.655952,0.181560,0.064057,79.690503,0.831573,Scored,True
SPACEL,0.188059,0.655706,0.183758,0.062148,83.686211,0.902289,Scored,True
STalign,0.185367,0.659121,0.183038,0.068431,73.721764,0.801403,Scored,True
PASTE2,NA,NA,NA,NA,NA,NA,Failed: 0/2 pairs,False
CAST,0.187550,0.645275,0.185057,0.061726,78.848240,0.894540,New: complete and scored,True
MOSAICField (RNA),0.128647,0.591948,0.129254,0.049071,101.993291,0.621480,New: complete and scored,True


... (2 outputs left out)

### E14_16

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.411842,0.947845,0.414820,0.191671,0.125625,0.643624,Scored,True
PASTE,0.382526,0.944291,0.386579,0.162621,0.136930,0.563724,Scored,True
SLAT,0.389036,0.944710,0.392797,0.165479,0.135331,0.579770,Scored,True
SPACEL,0.409362,0.945594,0.411990,0.192494,0.131897,0.635016,Scored,True
PASTE2,0.384870,0.939064,0.386243,0.171316,0.149774,0.592428,Scored,True
CAST,NA,NA,NA,NA,NA,0.126806,Native fallback; Region scored,True
MOSAICField (RNA),0.182823,0.914408,0.194066,0.089574,0.211897,0.372768,New: complete and scored,True
SANTO,0.321615,0.935610,0.327376,0.128812,0.157290,0.470798,New: complete and scored,True
STalign,0.385493,0.944777,0.389318,0.166618,0.134542,0.576439,New: complete and scored,True


### MLN

,PCC,CosSim,SSIM,MI,MSE,Region,Status,Eligible
OmniAligner,0.812007,0.982588,0.811243,0.809955,0.055572,0.406042,Scored,True
SLAT full sample,0.628273,0.955256,0.630293,0.410136,0.085370,0.263900,Scored,True
SPACEL,0.810401,0.981903,0.807871,0.797399,0.056288,0.406349,Scored,True
STalign,0.007397,0.148610,0.010834,0.010396,83.371461,0.032328,Scored,True
Spateo nonrigid,0.813540,0.980730,0.810787,0.734435,0.055805,0.378206,Scored,True
PASTE (5k),0.787607,0.979748,0.784433,0.586253,0.056072,0.429433,Scored,True
PASTE2 (5k),0.668936,0.961677,0.667826,0.430011,0.077474,0.306138,New: complete and scored,True
MOSAICField (RNA),0.671677,0.960303,0.671442,0.464766,0.082297,0.285989,New: complete and scored,True
SANTO,0.531558,0.939827,0.528887,0.306617,0.108158,0.236073,New: complete and scored,True
CAST,NA,NA,NA,NA,NA,NA,Incomplete; not ranked,False


## 3. Recompute algorithm ranks and denominators
Count each algorithm once, using its best eligible configuration for each metric. PASTE configurations count as PASTE, Spateo configurations as Spateo; PASTE2 remains separate. Undefined, failed, incomplete and diagnostic values do not enter that metric's denominator. Rank 1 is green; other ranks are gray.

In [3]:
def algorithm(name):
    for prefix in ["OmniAligner", "PASTE2", "PASTE", "Spateo", "SLAT", "MOSAICField"]:
        if name.startswith(prefix):
            return prefix
    return name
rank_rows = []
for dataset, group in rebuilt.groupby("Dataset", sort=False):
    for metric in METRICS:
        valid = group.loc[group.Eligible & group[metric].notna()].copy()
        valid["Algorithm"] = valid.Method.map(algorithm)
        scores = valid.groupby("Algorithm")[metric].agg("min" if metric == "MSE" else "max")
        value = scores["OmniAligner"]
        rank = 1 + int((scores < value).sum() if metric == "MSE" else (scores > value).sum())
        best = scores.min() if metric == "MSE" else scores.max()
        rank_rows.append({"Dataset": dataset, "Metric": metric, "OmniAligner score": value, "Rank": rank,
                          "Algorithms": len(scores), "Displayed rank": f"{rank}/{len(scores)}",
                          "Best algorithm(s)": "; ".join(sorted(scores.index[scores.eq(best)]))})
ranks = pd.DataFrame(rank_rows)
matrix = ranks.pivot(index="Dataset", columns="Metric", values="Displayed rank").reindex(index=rebuilt.Dataset.drop_duplicates(), columns=METRICS)
display_table(matrix, lambda row, col: "background:#d2e8dc;color:#225b43;font-weight:bold" if row[col].startswith("1/") else "background:#f1f3f4;color:#243447")
print("OmniAligner rank-one cells:", int(ranks.Rank.eq(1).sum()), "/", len(ranks))

,PCC,CosSim,SSIM,MI,MSE,Region
DLPFC,1/9,1/9,1/9,4/9,3/9,1/9
MHPR,1/9,1/9,1/9,1/9,1/9,2/9
MMPC,1/8,1/8,1/8,1/8,4/8,3/8
MPC,2/5,2/5,2/5,1/9,3/9,1/9
E14_16,1/8,1/8,1/8,2/8,1/8,1/9
MLN,2/9,1/9,1/9,1/9,1/9,3/9


OmniAligner rank-one cells: 24 / 36


## 4. Verify the entire submission table and ranking
The comparison checks all dataset–method rows, every numeric value and NA position, eligibility, status, ranks and denominators. Outputs are written to a separate directory; the reference table is not modified.

In [4]:
expected = pd.read_csv(SOURCE_ROOT / "submission_reference.csv")
assert rebuilt[["Dataset", "Method"]].values.tolist() == expected[["Dataset", "Method"]].values.tolist()
assert rebuilt.Eligible.tolist() == expected.Eligible.tolist()
assert rebuilt.Status.tolist() == expected.Status.tolist()
actual_values = rebuilt[METRICS].to_numpy(dtype=float)
expected_values = expected[METRICS].to_numpy(dtype=float)
assert np.array_equal(np.isnan(actual_values), np.isnan(expected_values)), "NA pattern differs"
assert np.allclose(actual_values, expected_values, rtol=0, atol=ATOL, equal_nan=True), "A table value differs"
expected_ranks = pd.read_csv(SOURCE_ROOT / "submission_ranks_reference.csv")
for column in ["Dataset", "Metric", "Rank", "Algorithms", "Displayed rank"]:
    assert ranks[column].tolist() == expected_ranks[column].tolist(), column
for actual, saved in zip(ranks["Best algorithm(s)"], expected_ranks["Best algorithm(s)"]):
    assert set(actual.split("; ")) == set(saved.split("; "))
rebuilt.to_csv(OUTPUT_ROOT / "NCS_submission_table_reproduced.csv", index=False)
ranks.to_csv(OUTPUT_ROOT / "NCS_submission_ranks_reproduced.csv", index=False)
report = {"all_table_entries_match": True, "all_rank_cells_match": True, "method_rows": len(rebuilt),
          "metric_positions": int(actual_values.size), "finite_metric_positions": int(np.isfinite(actual_values).sum()),
          "rank_cells": len(ranks), "absolute_tolerance": ATOL,
          "maximum_absolute_difference": float(np.nanmax(np.abs(actual_values - expected_values))),
          "pending": manifest["pending"], "snapshot": manifest["snapshot"],
          "comparison_scope": "entire submission candidate table and rank matrix",
          "evidence_modes": pd.Series([b["kind"] for b in manifest["bindings"]]).value_counts().to_dict()}
(OUTPUT_ROOT / "VALIDATION.json").write_text(json.dumps(report, indent=2))
display(pd.DataFrame([report]).drop(columns=["pending", "evidence_modes"]))
print("PASS: complete submission table and ranking reproduced.")
if manifest["pending"]:
    print("Pending rows remain unscored. Freeze the updated reference after completion; this snapshot is not a claim that those runs have finished.")

,all_table_entries_match,all_rank_cells_match,method_rows,metric_positions,finite_metric_positions,rank_cells,absolute_tolerance,maximum_absolute_difference,snapshot,comparison_scope
0,True,True,59,354,325,36,1.000000e-10,4.440892e-16,2026-11-17T01:07:58-07:00,entire submission candidate table and rank matrix


PASS: complete submission table and ranking reproduced.
